# Phase 15: ONNX vs PyTorch Parity Validation

This notebook runs parallel predictions on validation images using both the PyTorch classifier and the exported ONNX Runtime model. It validates classification logits agreement, error metrics, and asserts top-1 matching accuracy bounds.

In [1]:
# Setup paths and imports
from pathlib import Path
import sys
import torch
import torch.nn as nn
import numpy as np
import onnxruntime as ort
from PIL import Image
import time

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'ai' and PROJECT_ROOT.parent != PROJECT_ROOT:
    if (PROJECT_ROOT / 'ai').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'ai'
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils

In [2]:
# Load PyTorch Model
device = torch.device("cpu")
import torchvision.models as models
pytorch_model = models.efficientnet_b0()
in_features = pytorch_model.classifier[1].in_features
pytorch_model.classifier[1] = nn.Linear(in_features, utils.config.NUM_CLASSES)

best_pth = utils.paths.CLASSIFIER_MODEL_DIR / "best_model.pth"
if best_pth.exists():
    checkpoint = torch.load(str(best_pth), map_location=device)
    pytorch_model.load_state_dict(checkpoint['model_state_dict'])
pytorch_model.eval().to(device)

# Load ONNX Session
onnx_path = utils.paths.CLASSIFIER_MODEL_DIR / "classifier.onnx"
ort_session = ort.InferenceSession(str(onnx_path))

In [3]:
# Fetch test images and run comparison
test_images = list((utils.paths.CLASSIFICATION_DATASET_DIR / "val").glob("**/*.jpg"))[:50] + \
               list((utils.paths.CLASSIFICATION_DATASET_DIR / "val").glob("**/*.JPG"))[:50]

mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

pytorch_latencies = []
onnx_latencies = []
matches = 0
errors_ae = []

for img_path in test_images:
    # Preprocess Image
    img = Image.open(img_path).convert("RGB").resize((224, 224))
    img_arr = np.array(img).astype(np.float32) / 255.0
    img_arr = img_arr.transpose(2, 0, 1)  # HWC -> CHW
    img_normalized = (img_arr - mean) / std
    input_tensor = np.expand_dims(img_normalized, axis=0).astype(np.float32) # [1, 3, 224, 224]
    
    # PyTorch Inference
    with torch.no_grad():
        t_input = torch.from_numpy(input_tensor).to(device)
        t0 = time.perf_counter()
        pt_output = pytorch_model(t_input).cpu().numpy()
        pytorch_latencies.append(time.perf_counter() - t0)
        
    # ONNX Inference
    t0 = time.perf_counter()
    ort_inputs = {ort_session.get_inputs()[0].name: input_tensor}
    ort_output = ort_session.run(None, ort_inputs)[0]
    onnx_latencies.append(time.perf_counter() - t0)
    
    # Output comparison
    pt_class = np.argmax(pt_output[0])
    ort_class = np.argmax(ort_output[0])
    
    if pt_class == ort_class:
        matches += 1
        
    errors_ae.append(np.max(np.abs(pt_output[0] - ort_output[0])))

agreement_rate = (matches / len(test_images)) * 100 if test_images else 100.0
max_absolute_error = np.max(errors_ae) if errors_ae else 0.0
mean_absolute_error = np.mean(errors_ae) if errors_ae else 0.0

print(f"📊 Total Validation Images Tested: {len(test_images)}")
print(f"📊 Top-1 Class Agreement Rate   : {agreement_rate:.2f}%")
print(f"📊 Maximum Absolute Logit Error : {max_absolute_error:.8f}")
print(f"📊 Mean Absolute Logit Error    : {mean_absolute_error:.8f}")
print(f"📊 PyTorch Avg Latency (CPU)    : {np.mean(pytorch_latencies)*1000:.2f} ms")
print(f"📊 ONNX Avg Latency (CPU)       : {np.mean(onnx_latencies)*1000:.2f} ms")

# Enforce mathematical parity boundary rules
assert agreement_rate >= 95.0, f"Parity validation failed: Agreement rate is too low ({agreement_rate:.2f}%)"

📊 Total Validation Images Tested: 100
📊 Top-1 Class Agreement Rate   : 100.00%
📊 Maximum Absolute Logit Error : 0.00002480
📊 Mean Absolute Logit Error    : 0.00001390
📊 PyTorch Avg Latency (CPU)    : 40.39 ms
📊 ONNX Avg Latency (CPU)       : 9.37 ms
